# PathVQA fusion and LoRA experiments

Academic project portfolio edition. Uses a frozen ViT image encoder, GPT-2 with LoRA, and cross-attention fusion. Source: IIDS67692 coursework, adapted from a supplied lab. See README for attribution and evaluation limits.

Historical metrics are exported in ../results. Notebook outputs have been cleared; rerunning downloads data and model weights, then trains new models.


## 0. Environment, device, and configuration

In [ ]:
# =============================================================================
# IMPORTANT for Apple Silicon (MPS):
# The MPS fallback flag MUST be set BEFORE `import torch`. If torch initialises
# the MPS backend without it, any op MPS does not support raises an error
# instead of quietly falling back to the CPU.
# =============================================================================
import os
os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')         # unsupported ops -> run on CPU
# Keep PyTorch's default MPS memory limit.

import random, time, string, collections
import numpy as np, torch
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True   # a few PathVQA images are truncated; allow them to decode

# ---- Run configuration (edit these for your machine) -----------------------
SEED          = 42
NUM_SAMPLES   = 3000     # how many validation examples to subsample
VAL_FRACTION  = 0.2      # 80/20 train/test split of that subset
BATCH_SIZE    = 32       # Apple Silicon: use 8 (8 GB Mac) or 16 (16 GB+). 32 targets a CUDA GPU.
NUM_EPOCHS    = 5
LEARNING_RATE = 1e-4
MAX_TOK_LEN   = 64       # max tokens for the "question: ... answer: ..." training sequence
LORA_R, LORA_ALPHA, LORA_DROPOUT = 8, 16, 0.05
FUSION_HEADS  = 4        # attention heads inside the fusion block
MAX_NEW_TOKS  = 30       # answer-generation budget at inference
# ----------------------------------------------------------------------------

def set_seed(seed=SEED):
    """Make runs reproducible across Python, NumPy, and torch."""
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
set_seed()

# Device auto-detection: CUDA (Nuvolos/Colab) -> MPS (Apple Silicon) -> CPU
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')
print('Device:', DEVICE, '| torch', torch.__version__)

def empty_cache():
    """Free unused GPU memory between training runs (works for CUDA and MPS)."""
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()
    elif DEVICE.type == 'mps': torch.mps.empty_cache()

## Task 1 — PathVQA dataset and three example samples

PathVQA (He et al., 2020) pairs pathology images with question-answer pairs for clinical decision support, in both binary (yes/no) and open-ended formats. Answers are predominantly short and frequently a single word, which is important for interpreting the metrics later. We visualise three examples spanning the question-answer length spectrum: minimum, median, and maximum total length.

In [ ]:
# Load the PathVQA validation split.
# If this download stalls on an unauthenticated connection, set a token first:
#     os.environ['HF_TOKEN'] = 'hf_...'
from datasets import load_dataset
import matplotlib.pyplot as plt

pathvqa = load_dataset(
    'parquet',
    data_files={'validation': 'hf://datasets/flaviagiammarino/path-vqa/data/validation-*.parquet'},
    split='validation')
print('Validation samples:', len(pathvqa))

# Choose three examples by total (question + answer) length: shortest, median, longest.
qa_len = np.array([len(ex['question']) + len(ex['answer']) for ex in pathvqa])
demo_indices = [('min',    int(qa_len.argmin())),
                ('median', int(np.argsort(qa_len)[len(qa_len) // 2])),
                ('max',    int(qa_len.argmax()))]

# Show the three images with their reference question and answer.
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (tag, idx) in zip(axes, demo_indices):
    ex = pathvqa[idx]
    ax.imshow(ex['image']); ax.axis('off')
    ax.set_title(f"[{tag}] Q: {ex['question']}\nA: {ex['answer']}", fontsize=9, wrap=True)
plt.tight_layout(); plt.show()

## Dataloader (subset and 80/20 split)

Images are resized to 224x224 and normalised to the ViT-base/16 statistics (mean = std = 0.5) so they match the pretrained encoder.

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.transforms import InterpolationMode

class PathVQADataset(Dataset):
    """Wraps the HF dataset and returns (image_tensor, question_str, answer_str)."""
    def __init__(self, hf_dataset):
        self.dataset = hf_dataset
        # Resize + normalise to the ViT-base/16 statistics so inputs match the encoder.
        self.transform = transforms.Compose([
            transforms.Resize((224, 224), interpolation=InterpolationMode.BICUBIC),
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
        ])
    def __len__(self): return len(self.dataset)
    def __getitem__(self, i):
        s = self.dataset[i]
        return self.transform(s['image'].convert('RGB')), s['question'], s['answer']

# Subsample, then split 80/20. The two seeds make the subset and the split reproducible.
subset = pathvqa.shuffle(seed=SEED).select(range(NUM_SAMPLES))
split  = subset.train_test_split(test_size=VAL_FRACTION, seed=SEED)
train_ds, val_ds = PathVQADataset(split['train']), PathVQADataset(split['test'])

# num_workers=0 is required on macOS to avoid fork-related crashes.
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  drop_last=True, num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f'Train: {len(train_ds)}   Val: {len(val_ds)}')

## Tasks 2 and 3 — Cross-Attention and Gated Cross-Attention fusion

**Cross-Attention Fusion** (Vaswani et al., 2017): text tokens are queries and image patch tokens are keys and values, followed by a feed-forward network with residual connections and LayerNorm. Image information is written into the text stream from the first training step.

**Gated Cross-Attention Fusion** (Alayrac et al., 2022, Flamingo): the same attention, but each residual is scaled by a learnable `tanh` gate initialised to zero. Because tanh(0) = 0, the block is an identity map at initialisation, so the GPT-2 prior is preserved and the visual pathway opens only as gradients accumulate.

In [ ]:
import torch.nn as nn

class CrossAttentionFusion(nn.Module):
    """Plain cross-attention: text = query, image patches = key/value."""
    def __init__(self, hidden_dim=768, num_heads=8, dropout=0.1):
        super().__init__()
        self.cross_attn = nn.MultiheadAttention(hidden_dim, num_heads,
                                                dropout=dropout, batch_first=True)
        self.norm1, self.norm2 = nn.LayerNorm(hidden_dim), nn.LayerNorm(hidden_dim)
        self.ffn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 4), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim * 4, hidden_dim), nn.Dropout(dropout))

    def forward(self, text_embeds, image_embeds, text_mask=None):
        attended, _ = self.cross_attn(text_embeds, image_embeds, image_embeds,
                                      need_weights=False)
        x = self.norm1(text_embeds + attended)     # residual 1 + LayerNorm
        return self.norm2(x + self.ffn(x))         # residual 2 + LayerNorm


class GatedCrossAttentionFusion(nn.Module):
    """Pre-norm cross-attention with zero-initialised tanh gates (identity at init)."""
    def __init__(self, hidden_dim=768, num_heads=8, dropout=0.1):
        super().__init__()
        self.cross_attn = nn.MultiheadAttention(hidden_dim, num_heads,
                                                dropout=dropout, batch_first=True)
        self.norm1, self.norm2 = nn.LayerNorm(hidden_dim), nn.LayerNorm(hidden_dim)
        self.ffn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 4), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim * 4, hidden_dim), nn.Dropout(dropout))
        self.attn_gate = nn.Parameter(torch.zeros(1))   # tanh(0)=0 -> identity at start
        self.ffn_gate  = nn.Parameter(torch.zeros(1))

    def forward(self, text_embeds, image_embeds, text_mask=None):
        attended, _ = self.cross_attn(self.norm1(text_embeds), image_embeds, image_embeds,
                                      need_weights=False)
        x = text_embeds + torch.tanh(self.attn_gate) * attended            # gated residual 1
        return x + torch.tanh(self.ffn_gate) * self.ffn(self.norm2(x))     # gated residual 2

## Multimodal model (frozen ViT + LoRA GPT-2 + fusion)

`fusion_type` switches between the two fusion variants without changing any other code. Only the fusion block and the LoRA adapters are trainable; the ViT and the base GPT-2 weights stay frozen.

In [ ]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel, ViTModel
from peft import LoraConfig, TaskType, get_peft_model

class MedVQA(nn.Module):
    """Frozen ViT encoder + LoRA-adapted GPT-2 decoder + one cross-modal fusion block."""
    def __init__(self, fusion_type='cross', peft_config=None):
        super().__init__()
        # Frozen visual encoder (no gradients).
        self.visual_encoder = ViTModel.from_pretrained('google/vit-base-patch16-224-in21k')
        for p in self.visual_encoder.parameters():
            p.requires_grad = False

        # GPT-2 uses its EOS token as the pad token (standard practice).
        self.tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
        self.tokenizer.pad_token = self.tokenizer.eos_token

        # Wrap GPT-2 with LoRA so only small low-rank matrices are trained.
        gpt = GPT2LMHeadModel.from_pretrained('gpt2')
        self.gpt = get_peft_model(gpt, peft_config) if peft_config else gpt

        # Pick the fusion variant; everything else is identical between the two.
        fusion_cls = GatedCrossAttentionFusion if fusion_type == 'gated' else CrossAttentionFusion
        self.fusion = fusion_cls(hidden_dim=768, num_heads=FUSION_HEADS)
        self.fusion_type = fusion_type

    def encode_image(self, image):
        # ViT returns one token per patch plus a CLS token: shape [B, 197, 768].
        return self.visual_encoder(image).last_hidden_state

    def forward(self, image, input_ids, attention_mask, labels=None):
        image_embeds = self.encode_image(image)
        text_embeds  = self.gpt.get_input_embeddings()(input_ids)        # [B, T, 768]
        fused        = self.fusion(text_embeds, image_embeds, attention_mask)
        # Feed fused embeddings (not raw ids) into GPT-2; labels enable the LM loss.
        return self.gpt(inputs_embeds=fused, attention_mask=attention_mask, labels=labels)

In [ ]:
# LoRA targets GPT-2's fused QKV projection ('c_attn'); the bias is not adapted.
PEFT_CONFIG = LoraConfig(task_type=TaskType.CAUSAL_LM,
                         r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
                         target_modules=['c_attn'], bias='none')

# Sanity check: confirm only a small fraction of parameters are trainable.
_probe = MedVQA('cross', PEFT_CONFIG).to(DEVICE)
_tot = sum(p.numel() for p in _probe.parameters())
_tr  = sum(p.numel() for p in _probe.parameters() if p.requires_grad)
print(f'Total: {_tot/1e6:.1f}M | trainable: {_tr/1e6:.2f}M ({100*_tr/_tot:.2f}%)')
del _probe; empty_cache()

## Tokenisation and training loop

Each sample is encoded as `question: {q} answer: {a}` for training, with pad positions masked to -100 so they are ignored by the loss. The same loop trains both fusion variants.

In [ ]:
def make_train_batch(tokenizer, questions, answers, max_len=MAX_TOK_LEN):
    """Encode 'question: {q} answer: {a}<eos>' and build masked labels for training."""
    text = [f'question: {q} answer: {a}{tokenizer.eos_token}'
            for q, a in zip(questions, answers)]
    enc = tokenizer(text, padding=True, truncation=True, max_length=max_len,
                    return_tensors='pt')
    labels = enc.input_ids.clone()
    labels[enc.attention_mask == 0] = -100   # ignore PAD positions in the loss
    return enc.input_ids, enc.attention_mask, labels

In [ ]:
from torch.optim import AdamW

def train_model(fusion_type, num_epochs=NUM_EPOCHS, lr=LEARNING_RATE, log_every=20):
    """Train one model (fusion block + LoRA only) and return it with its loss history."""
    set_seed(SEED)                                  # identical init for a fair CA vs gated comparison
    model = MedVQA(fusion_type, PEFT_CONFIG).to(DEVICE)
    optim = AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=0.01)
    history, t0 = [], time.time()
    for epoch in range(num_epochs):
        model.train(); running = 0.0
        for step, (img, q, a) in enumerate(train_loader):
            ids, mask, labels = make_train_batch(model.tokenizer, q, a)
            img, ids, mask, labels = (t.to(DEVICE) for t in (img, ids, mask, labels))
            loss = model(img, ids, mask, labels=labels).loss
            optim.zero_grad(); loss.backward(); optim.step()
            running += loss.item()
            if step % log_every == 0:
                print(f'  [{fusion_type}] epoch {epoch+1} step {step:3d}/{len(train_loader)} '
                      f'loss {loss.item():.4f}')
        history.append(running / len(train_loader))
        print(f'>>> [{fusion_type}] epoch {epoch+1} mean loss {history[-1]:.4f} '
              f'({time.time()-t0:.0f}s)')
    return model, history

In [ ]:
# Train both fusion variants under identical settings.
# empty_cache() frees GPU memory between runs (important on Apple Silicon's shared memory).
print('=== Cross-Attention ===')
model_ca,  hist_ca  = train_model('cross'); empty_cache()
print('\n=== Gated Cross-Attention ===')
model_gca, hist_gca = train_model('gated'); empty_cache()

In [ ]:
# Training-loss curves as a quick sanity check that both models learned.
plt.figure(figsize=(5.5, 3.6))
plt.plot(range(1, len(hist_ca)+1),  hist_ca,  marker='o', label='Cross-Attention')
plt.plot(range(1, len(hist_gca)+1), hist_gca, marker='s', label='Gated Cross-Attention')
plt.xlabel('Epoch'); plt.ylabel('Training loss'); plt.title('Training loss per epoch')
plt.legend(frameon=False); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## Inference (greedy decoding through the fusion block)

We decode token by token, re-running the fusion and GPT-2 forward at each step. HuggingFace `generate` would bypass the fusion block, so a manual loop is required for correctness. Note: `nxt.item()` forces a GPU to CPU sync every token, which is slow on MPS; lower `MAX_EVAL` in the evaluation cell if a full pass takes too long on a Mac.

In [ ]:
@torch.no_grad()
def generate_answer(model, image, question, max_new_tokens=MAX_NEW_TOKS):
    """Greedy-decode one answer THROUGH the fusion block (cannot use .generate)."""
    model.eval(); tok = model.tokenizer
    enc = tok(f'question: {question} answer:', return_tensors='pt').to(DEVICE)
    ids, mask = enc.input_ids, enc.attention_mask
    image_embeds = model.encode_image(image.unsqueeze(0).to(DEVICE))   # encode the image once
    for _ in range(max_new_tokens):
        fused = model.fusion(model.gpt.get_input_embeddings()(ids), image_embeds, mask)
        nxt = model.gpt(inputs_embeds=fused, attention_mask=mask).logits[:, -1].argmax(-1, keepdim=True)
        if nxt.item() == tok.eos_token_id: break       # stop at end-of-sequence
        ids  = torch.cat([ids, nxt], 1)
        mask = torch.cat([mask, torch.ones_like(nxt)], 1)
    # Return only the text after 'answer:'.
    return tok.decode(ids[0], skip_special_tokens=True).split('answer:')[-1].strip()

## Evaluation metrics

We report the **three metrics required by the brief: BLEU, ROUGE-L, and METEOR**, plus two supplementary measures (exact-match Accuracy and Token-F1). BLEU is kept exactly as required, but note that for predominantly single-word answers it tends to 0.000 because there are no matching 4-grams; Accuracy and Token-F1 are therefore the more informative numbers for these short clinical answers.

In [ ]:
import nltk
for pkg in ['punkt', 'punkt_tab', 'wordnet', 'omw-1.4']:
    try: nltk.download(pkg, quiet=True)
    except Exception: pass
import evaluate

# --- the THREE metrics required by the brief ---
bleu_m   = evaluate.load('bleu')      # required
rouge_m  = evaluate.load('rouge')     # required (we report ROUGE-L)
meteor_m = evaluate.load('meteor')    # required

def _norm(s):
    """Lowercase, strip punctuation, collapse whitespace (used by Accuracy and F1)."""
    s = s.lower().strip().translate(str.maketrans('', '', string.punctuation))
    return ' '.join(s.split())

def _token_f1(pred, ref):
    """SQuAD-style token-overlap F1 between a prediction and its reference."""
    pt, rt = _norm(pred).split(), _norm(ref).split()
    if not pt or not rt: return float(pt == rt)
    common = sum((collections.Counter(pt) & collections.Counter(rt)).values())
    if common == 0: return 0.0
    prec, rec = common / len(pt), common / len(rt)
    return 2 * prec * rec / (prec + rec)

@torch.no_grad()
def evaluate_model(model, loader, max_new_tokens=MAX_NEW_TOKS, max_eval=None):
    """Generate answers over `loader` and return all metrics plus the raw preds/refs."""
    preds, refs = [], []
    for img, q, a in loader:
        for i in range(img.size(0)):
            preds.append(generate_answer(model, img[i], q[i], max_new_tokens) or '.')
            refs.append(a[i])
            if max_eval and len(preds) >= max_eval: break
        if max_eval and len(preds) >= max_eval: break
    return {
        # --- required metrics ---
        'BLEU':     round(bleu_m.compute(predictions=preds, references=[[r] for r in refs])['bleu'], 4),
        'ROUGE-L':  round(float(rouge_m.compute(predictions=preds, references=refs)['rougeL']), 4),
        'METEOR':   round(float(meteor_m.compute(predictions=preds, references=refs)['meteor']), 4),
        # --- supplementary (more informative for single-word answers) ---
        'Accuracy': round(float(np.mean([_norm(p) == _norm(r) for p, r in zip(preds, refs)])), 4),
        'Token-F1': round(float(np.mean([_token_f1(p, r) for p, r in zip(preds, refs)])), 4),
    }, preds, refs

## Task 5 — Hyperparameter tuning

We tune two hyperparameters. First the learning rate of the gated model, then, more substantially, the LoRA rank r. The rank controls both the capacity and the parameter count of the adaptation, so it is the central knob to study. The rank ablation below follows Hu et al. (2022), who swept several rank values to find where performance saturates.

In [ ]:
TUNE_LR, TUNE_EPOCHS = 5e-5, NUM_EPOCHS
print(f'=== Gated Cross-Attention (tuned: lr={TUNE_LR}, epochs={TUNE_EPOCHS}) ===')
model_tuned, hist_tuned = train_model('gated', num_epochs=TUNE_EPOCHS, lr=TUNE_LR); empty_cache()

### LoRA rank ablation

LoRA replaces a full weight update with a low-rank product `dW = B A`, where `B` has shape d x r and `A` has shape r x k, so the rank `r` is the bottleneck dimension and the number of trainable parameters grows linearly with `r`. We vary `r` while holding the LoRA scaling `alpha / r` constant at 2 (that is, `alpha = 2r`), so the comparison isolates the effect of rank rather than the scaling magnitude. The sweep runs on the plain Cross-Attention model, the stronger fusion variant, with every other setting fixed.

This cell trains one model per rank, so it is the most expensive in the notebook. Reduce `RANK_VALUES` or `RANK_MAX_EVAL` if compute is limited, and on Apple Silicon start small.

In [ ]:
import pandas as pd

RANK_VALUES   = [2, 4, 8, 16, 32]   # LoRA ranks to compare
RANK_MAX_EVAL = 200                 # eval samples per rank (set to None for the full 600)

def train_eval_with_rank(r, fusion_type='cross'):
    """Train one model at LoRA rank r (alpha = 2r) and return its params and metrics."""
    cfg = LoraConfig(task_type=TaskType.CAUSAL_LM,
                     r=r, lora_alpha=2 * r, lora_dropout=LORA_DROPOUT,   # alpha/r fixed at 2
                     target_modules=['c_attn'], bias='none')
    set_seed(SEED)
    model = MedVQA(fusion_type, cfg).to(DEVICE)
    optim = AdamW([p for p in model.parameters() if p.requires_grad],
                  lr=LEARNING_RATE, weight_decay=0.01)
    for _ in range(NUM_EPOCHS):
        model.train()
        for img, q, a in train_loader:
            ids, mask, labels = make_train_batch(model.tokenizer, q, a)
            img, ids, mask, labels = (t.to(DEVICE) for t in (img, ids, mask, labels))
            loss = model(img, ids, mask, labels=labels).loss
            optim.zero_grad(); loss.backward(); optim.step()
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    metrics, _, _ = evaluate_model(model, val_loader, max_eval=RANK_MAX_EVAL)
    del model; empty_cache()
    return {'rank': r, 'trainable_M': round(trainable / 1e6, 3), **metrics}

rank_rows = []
for r in RANK_VALUES:
    print(f'--- LoRA rank r = {r} (alpha = {2*r}) ---')
    row = train_eval_with_rank(r); print('  ', row)
    rank_rows.append(row)
rank_table = pd.DataFrame(rank_rows)
print('\nLoRA rank ablation (eval on', RANK_MAX_EVAL, 'samples)')
rank_table

In [ ]:
# Plot the ablation: does the metric keep rising with r, or saturate?
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(rank_table['rank'], rank_table['ROUGE-L'], marker='o', label='ROUGE-L')
ax.plot(rank_table['rank'], rank_table['METEOR'],  marker='s', label='METEOR')
ax.set_xscale('log', base=2)
ax.set_xticks(RANK_VALUES); ax.set_xticklabels([str(r) for r in RANK_VALUES])
ax.set_xlabel('LoRA rank r (log scale)'); ax.set_ylabel('Score')
ax.set_title('Effect of LoRA rank on PathVQA metrics')
ax.legend(frameon=False); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

## Figure 1 and Table 1

Table 1 reports the three required metrics (BLEU, ROUGE-L, METEOR) plus the two supplementary ones, one row per model. Figure 1 shows qualitative predictions: rows are the models (Task 2 Cross-Attention, Task 3 Gated, Task 5 Gated tuned) and columns are the three demo samples from Task 1.

In [ ]:
# Set MAX_EVAL = None to evaluate the full 600-sample test set.
# On Apple Silicon (MPS) token-by-token decoding is slow, so start with 200 to verify the
# pipeline, then set it to None for the final numbers.
MAX_EVAL = None
import pandas as pd

models = [('Cross-Attention Fusion', model_ca),
          ('Gated Cross-Attention Fusion', model_gca),
          (f'Gated CA (tuned, lr={TUNE_LR})', model_tuned)]

# Table 1: one row per model. Columns BLEU / ROUGE-L / METEOR are the required metrics.
table1_rows = []
for name, m in models:
    print('Evaluating:', name)
    metrics, _, _ = evaluate_model(m, val_loader, max_eval=MAX_EVAL)
    print('  ', metrics)
    table1_rows.append({'Model': name, **metrics})
table1 = pd.DataFrame(table1_rows)
print('\nTable 1 — VQA metrics (BLEU, ROUGE-L, METEOR are the required ones)')
table1

In [ ]:
# Figure 1: rows = models, columns = the three demo samples used in Task 1.
demo = [(tag, pathvqa[idx], train_ds.transform(pathvqa[idx]['image'].convert('RGB')))
        for tag, idx in demo_indices]

fig, axes = plt.subplots(len(models), 3, figsize=(15, 4.6 * len(models)))
for r, (name, m) in enumerate(models):
    for c, (tag, ex, img_t) in enumerate(demo):
        ax = axes[r, c]
        ax.imshow(ex['image']); ax.axis('off')
        pred = generate_answer(m, img_t, ex['question'])   # model's predicted answer
        ax.set_title(f"{name.split('(')[0].strip()}\n[{tag}] Q: {ex['question']}\n"
                     f"Ref: {ex['answer']}  |  Pred: {pred}", fontsize=8, loc='left', wrap=True)
fig.suptitle('Figure 1 — qualitative PathVQA predictions', y=1.001, fontsize=12)
plt.tight_layout(); plt.show()

## Discussion checked against the saved outputs

The saved 600-example evaluation gives ROUGE-L 0.3483 for plain cross-attention, 0.3175 for gated cross-attention, and 0.2559 for the gated model with learning rate 5e-5. Lowering the learning rate did not improve this run. A single run does not establish the mechanism behind the differences.

The separate rank sweep uses the first 200 examples of the evaluation subset, with alpha/r fixed at 2. Rank 16 scores 0.3902 ROUGE-L; rank 32 scores 0.3802. These are within-project subset results, not official PathVQA benchmark results. Rank and model configuration selection use evaluation feedback; an untouched test split is needed for a final performance claim.

BLEU-4 is zero in the saved outputs; exact match and token F1 help interpret the predominantly short answers. The training loss includes question as well as answer tokens; it is not an answer-only objective.


## Save outputs for the report

In [ ]:
import json as _json
table1.to_csv('table1_metrics.csv', index=False)
rank_table.to_csv('rank_ablation.csv', index=False)
with open('partB_outputs.json', 'w') as f:
    _json.dump({'table1': table1.to_dict(orient='records'),
                'rank_ablation': rank_table.to_dict(orient='records')}, f, indent=2)
print('Saved table1_metrics.csv, rank_ablation.csv, and partB_outputs.json')